In [1]:
import pandas as pd
import os

# --- 配置区 ---
dataset_list = ['X-Topic']
F_NUM = 5
P_NUM = 0
KNOWN_LABEL_RATIOS = [0.25, 0.5, 0.75]
LABELED_DATA_RATIOS = [0.1, 0.5, 1.0]

# --- 统计容器：显式区分三维 ---
rows = []
print("--- Step 2, Block 1: Calculating all statistics ---")

for dataset_name in dataset_list:
    try:
        # 整体样本量 / 标签总数（不区分 rate/LDR）
        total_train = len(pd.read_csv(f'{dataset_name}/origin_data/train.tsv', sep='\t'))
        total_dev   = len(pd.read_csv(f'{dataset_name}/origin_data/dev.tsv',   sep='\t'))
        total_test  = len(pd.read_csv(f'{dataset_name}/origin_data/test.tsv',  sep='\t'))
        total_labels= len(pd.read_csv(f'{dataset_name}/label/label.list', header=None))

        # rows.append(dict(dataset=dataset_name, known_label_ratio='ALL', labeled_data_ratio='ALL', split='train',  value=total_train))
        # rows.append(dict(dataset=dataset_name, known_label_ratio='ALL', labeled_data_ratio='ALL', split='dev',    value=total_dev))
        # rows.append(dict(dataset=dataset_name, known_label_ratio='ALL', labeled_data_ratio='ALL', split='test',   value=total_test))
        # rows.append(dict(dataset=dataset_name, known_label_ratio='ALL', labeled_data_ratio='ALL', split='\\#label', value=total_labels))

        # --- 在遍历 rate 之前，先为每个 LDR 写入 "满 known (ALL)" 的 train/dev/test/#label ---
        for LDR in LABELED_DATA_RATIOS:
            # \#label：总类数（不受 LDR 影响）
            rows.append(dict(
                dataset=dataset_name,
                known_label_ratio='ALL',
                labeled_data_ratio=LDR,
                split='\\#label',
                value=total_labels
            ))

            # train/dev：读取对应 LDR 的 labeled_data（若无 'labeled' 列则全计）
            for split in ['train', 'dev']:
                ldr_path = f'{dataset_name}/labeled_data/{LDR}/{split}.tsv'
                if os.path.exists(ldr_path):
                    df_ldr = pd.read_csv(ldr_path, sep='\t')
                    if 'labeled' in df_ldr.columns:
                        cnt = int((df_ldr['labeled'] == True).sum())
                    else:
                        cnt = len(df_ldr)
                    rows.append(dict(
                        dataset=dataset_name,
                        known_label_ratio='ALL',
                        labeled_data_ratio=LDR,
                        split=split,
                        value=cnt
                    ))
                # 若当前 LDR 缺该 split 文件则不写入（避免产生空列，后续本就会 drop 全空列）

            # test：若 LDR 下没有 test 文件，则回退到 origin_data 的 test 总数
            ldr_test_path = f'{dataset_name}/labeled_data/{LDR}/test.tsv'
            if os.path.exists(ldr_test_path):
                df_test_ldr = pd.read_csv(ldr_test_path, sep='\t')
                # test 一般无 labeled 列，直接全计
                test_cnt = len(df_test_ldr)
            else:
                test_cnt = total_test  # 回退
            rows.append(dict(
                dataset=dataset_name,
                known_label_ratio='ALL',
                labeled_data_ratio=LDR,
                split='test',
                value=test_cnt
            ))

        # 分 rate 统计
        for rate in KNOWN_LABEL_RATIOS:
            known_label_path = f'{dataset_name}/label/fold{F_NUM}/part{P_NUM}/label_known_{rate}.list'
            if not os.path.exists(known_label_path):
                print(f"    [警告] 缺少 {known_label_path}，跳过该 rate")
                continue
            known_labels = pd.read_csv(known_label_path, header=None)[0].tolist()
            known_cnt = len(known_labels)

            # 1) 在每个 LDR 下都写入一条 '#label'（即使没有 train/dev 文件也会显示）
            for LDR in LABELED_DATA_RATIOS:
                rows.append(dict(
                    dataset=dataset_name,
                    known_label_ratio=rate,
                    labeled_data_ratio=LDR,
                    split='\\#label',
                    value=known_cnt
                ))

                # 2) 分 LDR & split 的“已标注且属于已知类”的样本数
                for split in ['train', 'dev']:
                    labeled_data_path = f'{dataset_name}/labeled_data/{LDR}/{split}.tsv'
                    if not os.path.exists(labeled_data_path):
                        continue
                    df_labeled = pd.read_csv(labeled_data_path, sep='\t')
                    if 'labeled' in df_labeled.columns:
                        mask = df_labeled['label'].isin(known_labels) & (df_labeled['labeled'] == True)
                    else:
                        mask = df_labeled['label'].isin(known_labels)
                    rows.append(dict(
                        dataset=dataset_name,
                        known_label_ratio=rate,
                        labeled_data_ratio=LDR,
                        split=split,
                        value=int(mask.sum())
                    ))

    except FileNotFoundError as e:
        print(f"    [错误] 统计失败，缺少文件: {e.filename}，已跳过 {dataset_name}")
        continue

print("--- Data calculation complete. ---")
df = pd.DataFrame(rows)
df

--- Step 2, Block 1: Calculating all statistics ---
--- Data calculation complete. ---


,dataset,known_label_ratio,labeled_data_ratio,split,value
0,X-Topic,ALL,0.1,\#label,49
1,X-Topic,ALL,0.1,train,610
2,X-Topic,ALL,0.1,dev,112
3,X-Topic,ALL,0.1,test,1767
4,X-Topic,ALL,0.5,\#label,49
5,X-Topic,ALL,0.5,train,2982
6,X-Topic,ALL,0.5,dev,444
7,X-Topic,ALL,0.5,test,1767
8,X-Topic,ALL,1.0,\#label,49
9,X-Topic,ALL,1.0,train,5954


In [2]:
# --- 构建 DataFrame 并做嵌套透视（先 LDR 再 rate） ---

pivot = df.pivot_table(
    index='dataset',
    columns=['labeled_data_ratio', 'known_label_ratio', 'split'],  # 顺序：LDR -> rate -> split
    values='value',
    aggfunc='sum'
)

# 1) 删除全为空值的列
pivot = pivot.dropna(axis=1, how='all')

# 2) 各层排序：数字在前，'ALL' 在后；split 顺序为 #label, train, dev, test
def _sort_key_num_all(v):
    if v == 'ALL':
        return (1, float('inf'))
    try:
        return (0, float(v))
    except:
        return (0, 0)

# 重新按排序后的多级列重建列索引（只包含目前存在的列）
lvl0_vals = sorted(pivot.columns.get_level_values(0).unique(), key=_sort_key_num_all)  # LDR
lvl1_vals = sorted(pivot.columns.get_level_values(1).unique(), key=_sort_key_num_all)  # rate
# split 层：仅保留现有的，并按优先序排列
split_current = list(pivot.columns.get_level_values(2).unique())
split_order = ['\\#label', 'train', 'dev', 'test']
lvl2_vals = [s for s in split_order if s in split_current] + [s for s in split_current if s not in split_order]

# 只对存在的组合进行笛卡尔积（避免引入空列）
from itertools import product
kept_cols = []
existing = set(pivot.columns)
for a, b, c in product(lvl0_vals, lvl1_vals, lvl2_vals):
    col = (a, b, c)
    if col in existing:
        kept_cols.append(col)

pivot = pivot.reindex(columns=pd.MultiIndex.from_tuples(kept_cols, names=pivot.columns.names))

# 3) 行也做一个自然排序
pivot = pivot.reindex(index=sorted(pivot.index))

# 4) 数字千分位格式化（保留 NaN）
pivot_fmt = pivot.applymap(lambda x: "{:,}".format(int(x)) if pd.notnull(x) else x)

pivot_fmt.to_excel('data_statics_nested.xlsx')
print("Saved to data_statics_nested.xlsx")
pivot_fmt

Saved to data_statics_nested.xlsx


/tmp/ipykernel_248275/1747660759.py:45: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pivot_fmt = pivot.applymap(lambda x: "{:,}".format(int(x)) if pd.notnull(x) else x)


labeled_data_ratio     0.1                                                 \
known_label_ratio     0.25               0.5              0.75              
split              \#label train dev \#label train dev \#label train  dev   
dataset                                                                     
X-Topic                 12   531  75      24   582  87      37   598  100   

labeled_data_ratio          ...     1.0                                  \
known_label_ratio      ALL  ...     0.5                0.75               
split              \#label  ... \#label  train  dev \#label  train  dev   
dataset                     ...                                           
X-Topic                 49  ...      24  5,836  825      37  5,937  849   

labeled_data_ratio                             
known_label_ratio      ALL                     
split              \#label  train  dev   test  
dataset                                        
X-Topic                 49  5,954  866  1,767  

[1 rows x 39 columns]

In [3]:
# --- 透视：行= (dataset, labeled_data_ratio)，列= (known_label_ratio, split) ---
pivot = df.pivot_table(
    index=['labeled_data_ratio', 'dataset'],
    columns=['known_label_ratio', 'split'],
    values='value',
    aggfunc='sum'
)

# 1) 删除全为空值的列
pivot = pivot.dropna(axis=1, how='all')

# 2) 各层排序：数字在前，'ALL' 在后；split 顺序为 #label, train, dev, test
def _sort_key_num_all(v):
    if v == 'ALL':
        return (1, float('inf'))
    try:
        return (0, float(v))
    except Exception:
        return (0, 0)

# 2.1 列排序（列两层：known_label_ratio, split）
if isinstance(pivot.columns, pd.MultiIndex):
    # level 0: known_label_ratio
    lvl0_vals = sorted(pivot.columns.get_level_values(0).unique(), key=_sort_key_num_all)
    # level 1: split
    split_current = list(pivot.columns.get_level_values(1).unique())
    split_order = ['\\#label', 'train', 'dev', 'test']
    lvl1_vals = [s for s in split_order if s in split_current] + [s for s in split_current if s not in split_order]

    # 仅保留现有组合，按期望顺序重建
    from itertools import product
    kept_cols = []
    existing = set(pivot.columns)
    for a, b in product(lvl0_vals, lvl1_vals):
        col = (a, b)
        if col in existing:
            kept_cols.append(col)
    if kept_cols:
        pivot = pivot.reindex(columns=pd.MultiIndex.from_tuples(kept_cols, names=pivot.columns.names))

# 2.2 行排序（行两层：dataset, labeled_data_ratio；其中 labeled_data_ratio 按数字在前、'ALL' 在后）
def _sort_key_ldr(v):
    return _sort_key_num_all(v)

if isinstance(pivot.index, pd.MultiIndex):
    # 取出现有行索引，按 (dataset 字典序, labeled_data_ratio 的数字/ALL 规则) 排序
    idx_list = list(pivot.index)
    idx_list_sorted = sorted(idx_list, key=lambda t: (t[0], _sort_key_ldr(t[1])))
    pivot = pivot.reindex(index=idx_list_sorted)
else:
    # 单层索引兜底：自然排序
    pivot = pivot.sort_index()

# 3) 千分位格式化（保留 NaN）
pivot_fmt = pivot.applymap(lambda x: "{:,}".format(int(x)) if pd.notnull(x) else x)

# 4) 导出
pivot_fmt.to_excel('data_statics_nested.xlsx')
print("Saved to data_statics_nested.xlsx")

pivot_fmt

Saved to data_statics_nested.xlsx


/tmp/ipykernel_248275/1797272048.py:55: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pivot_fmt = pivot.applymap(lambda x: "{:,}".format(int(x)) if pd.notnull(x) else x)


known_label_ratio             0.25                 0.5                0.75  \
split                      \#label  train  dev \#label  train  dev \#label   
labeled_data_ratio dataset                                                   
0.1                X-Topic      12    531   75      24    582   87      37   
0.5                X-Topic      12  2,657  378      24  2,917  415      37   
1.0                X-Topic      12  5,318  756      24  5,836  825      37   

known_label_ratio                          ALL                     
split                       train  dev \#label  train  dev   test  
labeled_data_ratio dataset                                         
0.1                X-Topic    598  100      49    610  112  1,767  
0.5                X-Topic  2,970  432      49  2,982  444  1,767  
1.0                X-Topic  5,937  849      49  5,954  866  1,767

In [4]:
data_statics = pivot_fmt.to_latex().replace(' ', '').replace('0000', '')
replace_map = {
    "Classes": '|Classes|',
    "train": '|Train|',
    "dev": '|Validation|',
    "test": '|Test|',
    "0000": "",
    ".00": "",
    # "lrrrrrrr": "l|p{1.2cm}p{1.2cm}p{1.2cm}p{1.2cm}p{1.2cm}p{1.2cm}p{1.2cm}",
    "toprule\n": "toprule\nDataset",
    "\nlabeled_data_ratio&dataset&&&&&&&&&&&&&\\\\": "",
    "known_label_ratio": "KCR",
    "labeled_data_ratio": "LAR",
    "split": "",
    " ": ""
}
for i, v in replace_map.items():
    data_statics = data_statics.replace(i, v)
with open('data_statics.latex', 'w') as w:
    w.write(data_statics)

In [5]:
import pandas as pd
import os

# 读取 dataset -> language/domain 的映射表
df_type = pd.read_csv("data_type.csv")   # 确保文件和脚本在同一目录，或写绝对路径

rows = []

for _, row in df_type.iterrows():
    dataset_name = row['dataset']
    Dataset_name = row['Dataset']
    language = row['Language']
    domain = row['Text Domain']
    URL = row['URL']
    Scenario = row['Scenario']

    try:
        base_path = os.path.join(dataset_name, "origin_data")
        train_path = os.path.join(base_path, "train.tsv")
        dev_path   = os.path.join(base_path, "dev.tsv")
        test_path  = os.path.join(base_path, "test.tsv")

        # 合并 train/dev/test
        df_all = pd.concat([
            pd.read_csv(train_path, sep="\t"),
            pd.read_csv(dev_path, sep="\t"),
            pd.read_csv(test_path, sep="\t")
        ], ignore_index=True)

        # 标签数量
        num_labels = df_all['label'].nunique()

        # 平均字符长度
        avg_len = df_all['text'].astype(str).str.len().mean()
        avg_label_len = df_all['label'].astype(str).str.len().mean()

        rows.append({
            'Task': Scenario,
            'Text Domain': domain,
            'Dataset': Dataset_name,
            'Lang': language,
            '\#Instance': len(df_all),
            '\#Label': num_labels,
            'Avg. Instance': avg_len.round(2),
            'Avg. Label': avg_label_len.round(2),
            # 'Url': URL
        })

    except FileNotFoundError as e:
        print(f"[错误] 缺少文件 {e.filename}, 已跳过 {dataset_name}")
        continue

# 汇总表
df_stats = pd.DataFrame(rows).set_index('Task')
latex_code = df_stats.to_latex()
print(latex_code.replace('_', '\_').replace('0000', '').replace('& 1 &', '& 10000 &'))

<>:42: SyntaxWarning: invalid escape sequence '\#'
<>:43: SyntaxWarning: invalid escape sequence '\#'
<>:56: SyntaxWarning: invalid escape sequence '\_'
<>:42: SyntaxWarning: invalid escape sequence '\#'
<>:43: SyntaxWarning: invalid escape sequence '\#'
<>:56: SyntaxWarning: invalid escape sequence '\_'
/tmp/ipykernel_248275/2697623239.py:42: SyntaxWarning: invalid escape sequence '\#'
  '\#Instance': len(df_all),
/tmp/ipykernel_248275/2697623239.py:43: SyntaxWarning: invalid escape sequence '\#'
  '\#Label': num_labels,
/tmp/ipykernel_248275/2697623239.py:56: SyntaxWarning: invalid escape sequence '\_'
  print(latex_code.replace('_', '\_').replace('0000', '').replace('& 1 &', '& 10000 &'))


\begin{tabular}{llllrrrr}
\toprule
 & Text Domain & Dataset & Lang & \#Instance & \#Label & Avg. Instance & Avg. Label \\
Task &  &  &  &  &  &  &  \\
\midrule
Topic Classification & Usenet News & 20NG & EN & 10000 & 20 & 1856.27 & 15.60 \\
Topic Classification & Sina News & THUCNews & CN & 9421 & 14 & 925.92 & 2.00 \\
Topic Classification & Yahoo pages & Yahoo & EN & 10000 & 10 & 525.19 & 17.30 \\
Intent Detection & Online banking queries & BANKING77 & EN & 13083 & 77 & 58.24 & 20.96 \\
Intent Detection & Conversational queries & CLINC150 & EN & 22500 & 150 & 39.78 & 11.79 \\
Intent Detection & Programming queries & StackOverflow & EN & 19985 & 20 & 50.51 & 6.00 \\
Intent Detection & Conversational queries & HWU64 & EN & 9677 & 64 & 33.53 & 13.72 \\
Intent Detection & Fact-based questiont & TREC & EN & 5871 & 47 & 49.46 & 10.03 \\
Intent Detection & Conversational queries & ECDT & CN & 3069 & 31 & 8.39 & 5.70 \\
Intent Detection & COVID-19 utterance & MCID & EN & 1699 & 16 & 35.62 & 1